# 02 — From Spinor to Bloch Sphere

A single-qubit state vector is called a **spinor**.  The Bloch sphere is a way to visualise it geometrically.

This notebook shows how the map works.

---

## The spinor

A normalised single-qubit state is:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad |\alpha|^2 + |\beta|^2 = 1$$

Using the **global phase freedom**, we can always write:

$$|\psi\rangle = \cos\!\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\!\frac{\theta}{2}|1\rangle$$

where $\theta \in [0, \pi]$ and $\phi \in [0, 2\pi)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere
from helpers.display_utils import show_state_vector, show_info_table
setup_notebook()


## The Bloch vector

Given angles $(\theta, \phi)$, the **Bloch vector** is the point on the unit sphere:

$$\vec{r} = (\sin\theta\cos\phi,\; \sin\theta\sin\phi,\; \cos\theta)$$

This gives a one-to-one map between *physical* (ray) states and points on the 2-sphere.


In [ ]:
def spinor_to_bloch(alpha, beta):
    """
    Convert a normalised spinor (alpha, beta) to a Bloch vector (x, y, z).
    NOTE: In production use rqm-core for this.
    """
    alpha, beta = complex(alpha), complex(beta)
    # Normalise
    norm = np.sqrt(abs(alpha)**2 + abs(beta)**2)
    alpha, beta = alpha / norm, beta / norm

    x = 2 * (alpha.conjugate() * beta).real
    y = 2 * (alpha.conjugate() * beta).imag
    z = abs(alpha)**2 - abs(beta)**2
    return np.array([x, y, z])

# Standard basis states
states = {
    r"|0⟩": (1, 0),
    r"|1⟩": (0, 1),
    r"|+⟩": (1/np.sqrt(2), 1/np.sqrt(2)),
    r"|-⟩": (1/np.sqrt(2), -1/np.sqrt(2)),
    r"|+i⟩": (1/np.sqrt(2), 1j/np.sqrt(2)),
}

print(f"{'State':<8} {'x':>8} {'y':>8} {'z':>8}")
print("-" * 38)
for label, (a, b) in states.items():
    bv = spinor_to_bloch(a, b)
    print(f"{label:<8} {bv[0]:>8.4f} {bv[1]:>8.4f} {bv[2]:>8.4f}")


In [ ]:
# Visualise all standard states on the Bloch sphere
vectors = []
colours = ["green", "red", "steelblue", "darkorange", "violet"]
for (label, (a, b)), colour in zip(states.items(), colours):
    vectors.append((spinor_to_bloch(a, b), label, colour))

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
draw_bloch_sphere(ax=ax, vectors=vectors, title="Standard basis states on the Bloch sphere")
plt.tight_layout()
plt.show()


## Measurement probabilities from the Bloch vector

Given a Bloch vector $\vec{r} = (x, y, z)$:

- $P(|0\rangle) = \frac{1 + z}{2}$
- $P(|1\rangle) = \frac{1 - z}{2}$

The north pole $z=1$ is $|0\rangle$ with certainty; the south pole $z=-1$ is $|1\rangle$ with certainty; the equator is a superposition.


In [ ]:
def measurement_probs(bloch_vector):
    z = bloch_vector[2]
    return (1 + z) / 2, (1 - z) / 2

print(f"{'State':<8} {'P(|0⟩)':>10} {'P(|1⟩)':>10}")
print("-" * 32)
for label, (a, b) in states.items():
    bv = spinor_to_bloch(a, b)
    p0, p1 = measurement_probs(bv)
    print(f"{label:<8} {p0:>10.4f} {p1:>10.4f}")


## Summary

- A spinor $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ maps to the Bloch vector $(x,y,z)$.
- North pole = $|0\rangle$, South pole = $|1\rangle$, equator = equal superposition.
- Measurement probabilities are $\frac{1 \pm z}{2}$.
- In `rqm-core`, `Spinor.to_bloch()` performs this conversion for you.

**Next:** [03_su2_rotations_and_geometry.ipynb](03_su2_rotations_and_geometry.ipynb)
